In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
stop_words = set("a an the is are was were be been am i me my we you your it its of to in on at for with and or but if do does did can could should would will what how why when which who whom this that these those there about from by as so than too very just have has had into over after before during up down out".split())

def preprocess(text):
    words = re.findall(r"[a-z]+", str(text).lower())
    words = [w for w in words if w not in stop_words]
    try:
        words = [lemmatizer.lemmatize(w) for w in words]
    except LookupError:
        words = [w[:-1] if len(w) > 4 and w.endswith("s") else w for w in words]
    return " ".join(words)

In [ ]:
data = {
"symptoms": [
"What are the symptoms of diabetes", "What are early warning signs of diabetes", "Why am I always thirsty", "Why do I urinate so often", "Can diabetes cause blurry vision", "Why am I tired all the time", "Can diabetes cause unexplained weight loss", "Why do I feel hungry after eating", "Do wounds heal slowly with diabetes", "Can diabetes cause frequent infections", "My mouth is always dry could it be diabetes", "I feel thirsty and keep going to the bathroom", "What does high blood sugar feel like", "Can tingling in feet be a symptom", "Do symptoms of type 1 diabetes appear quickly", "Can type 2 diabetes have no symptoms", "Why is my vision suddenly blurry", "Is extreme hunger a sign of high glucose", "What are common signs of low blood sugar", "What happens when blood sugar becomes high", "Can diabetes make me feel weak", "Are headaches related to blood sugar", "What are the first signs someone has diabetes", "I am losing weight and feel thirsty", "Can diabetes cause itching"
],
"risk_factors": [
"What causes type 2 diabetes", "Does diabetes run in families", "Does family history increase my risk", "Can obesity increase diabetes risk", "Does being overweight cause diabetes", "Can lack of exercise lead to diabetes", "Does age increase the risk of diabetes", "Is high blood pressure a risk factor", "Does prediabetes turn into diabetes", "Can gestational diabetes increase future risk", "Does eating too much sugar cause diabetes", "Who is more likely to develop type 2 diabetes", "Can PCOS increase diabetes risk", "Does belly fat increase the risk", "Can genetics cause diabetes", "Is diabetes more common after age 45", "Does smoking increase diabetes risk", "Can an inactive lifestyle increase risk", "Does having a parent with diabetes matter", "What are the main risk factors", "Can sleep problems affect diabetes risk", "Is high cholesterol linked to diabetes risk", "Can pregnancy cause diabetes", "How can I reduce my risk of diabetes", "Am I at risk if diabetes runs in my family"
],
"diet": [
"What should a person with diabetes eat", "Which foods help control blood sugar", "Can diabetics eat fruit", "Should I avoid sugary drinks", "Is brown rice better than white rice", "Can people with diabetes eat bread", "What is a good breakfast for diabetes", "Are oats good for blood sugar", "Can I eat bananas with diabetes", "Should diabetics avoid potatoes", "What foods should I limit", "Is whole wheat better than refined flour", "Can I drink fruit juice", "Are lentils good for diabetes", "What is a balanced diabetic meal", "How many carbohydrates should I eat", "Can I eat rice if I have diabetes", "Are nuts a healthy snack for diabetics", "Does cinnamon control blood sugar", "What are healthy snacks for diabetes", "Should I skip meals to lower glucose", "Is honey safe for people with diabetes", "What vegetables are good for diabetes", "Can a diabetic person eat sweets sometimes", "How can I plan meals to manage glucose"
],
"exercise": [
"Is walking good for diabetes", "Can exercise lower blood sugar", "How often should diabetics exercise", "Is yoga helpful for diabetes", "Can I go cycling with diabetes", "Does walking after meals help glucose", "What exercise is best for type 2 diabetes", "Can strength training help diabetes", "How long should I walk each day", "Is swimming safe for diabetics", "Can exercise improve insulin sensitivity", "Should I check glucose before exercise", "Can physical activity prevent type 2 diabetes", "Is running suitable for people with diabetes", "What if my sugar drops during exercise", "Can I work out with high blood sugar", "Does sitting all day affect diabetes", "Are short walks useful for glucose control", "Can resistance exercise help blood sugar", "How many days a week should I be active", "Is dancing a good exercise for diabetes", "Can exercise help with weight management", "Should I carry glucose during workouts", "What precautions should diabetics take while exercising", "Can regular activity reduce insulin resistance"
],
"medication": [
"What is metformin used for", "How does insulin work", "What medicines treat type 2 diabetes", "Is metformin a diabetes medicine", "Why do some diabetics need insulin", "What are common side effects of metformin", "Can diabetes medicine cause low blood sugar", "What is the difference between insulin and metformin", "Are tablets enough to manage diabetes", "Can I stop taking my diabetes medication", "When should diabetes medicines be taken", "What are the types of insulin", "Do all people with diabetes need insulin", "Can medicines cure diabetes", "What should I know about diabetes injections", "Why has my doctor prescribed metformin", "Can I take diabetes medicine with food", "What happens if I miss a dose", "Are there side effects of insulin", "How do GLP 1 medicines help diabetes", "What is an oral diabetes medication", "Can I change my insulin dose myself", "Which medicines lower blood glucose", "How is type 1 diabetes treated", "How is type 2 diabetes treated with medicine"
],
"glucose_monitoring": [
"How do I check my blood sugar", "What is an HbA1c test", "What does HbA1c measure", "How often should I monitor glucose", "What is a normal fasting blood sugar reading", "When should I test blood glucose", "How does a glucose meter work", "What is a continuous glucose monitor", "What does a post meal glucose test show", "Why is my fasting sugar high", "What is the target blood sugar range", "How do I use a glucometer", "What does a high HbA1c mean", "How often is HbA1c checked", "Can I check glucose at home", "What is the difference between fasting glucose and HbA1c", "When should blood sugar be checked after eating", "Why do glucose readings change during the day", "What is a glucose sensor", "How do I record my sugar readings", "What does low blood glucose mean", "What test confirms diabetes", "Can stress affect glucose readings", "What is an oral glucose tolerance test", "How can I understand my blood sugar report"
],
"complications": [
"What complications can diabetes cause", "Can diabetes damage the kidneys", "How does diabetes affect eyesight", "Can high blood sugar damage nerves", "What is diabetic neuropathy", "Can diabetes cause heart disease", "Why is foot care important for diabetics", "Can diabetes cause kidney failure", "What is diabetic retinopathy", "Does diabetes affect wound healing", "Can diabetes cause numb feet", "How can I prevent diabetes complications", "Can diabetes affect the heart", "Why should diabetics get eye checkups", "What happens if diabetes is uncontrolled", "Can diabetes cause gum disease", "Does diabetes increase stroke risk", "What are long term effects of high glucose", "Can diabetes damage blood vessels", "Why do diabetics need regular health checks", "Can diabetes affect sexual health", "What is diabetic foot", "Can nerve damage from diabetes be prevented", "How does diabetes affect the eyes and kidneys", "What organs can diabetes affect"
],
"emergency": [
"What should I do if someone with diabetes is unconscious", "A diabetic person is having a seizure", "What are signs of diabetic ketoacidosis", "What should I do for severe low blood sugar", "My friend with diabetes passed out", "When is high blood sugar an emergency", "What are symptoms of a diabetic emergency", "Someone with diabetes is unresponsive", "What should I do if a diabetic cannot swallow", "Can diabetic ketoacidosis be life threatening", "What are warning signs of severe hypoglycemia", "A person with diabetes is confused and fainting", "What should I do if someone has fruity breath and vomiting", "When should I call emergency services for diabetes", "My blood sugar is extremely low and I feel faint", "What are signs of hyperglycemic crisis", "A diabetic person is breathing rapidly and vomiting", "What if a person with diabetes loses consciousness", "How do I respond to a diabetes seizure", "Is severe hypoglycemia dangerous", "What should I do if someone cannot wake up", "Can very high glucose cause an emergency", "A diabetic person is shaking sweating and confused", "When does low blood sugar need urgent help", "What are symptoms of hyperosmolar hyperglycemic state"
]
}

corpus = [(question, intent) for intent, questions in data.items() for question in questions]
print("Total queries:", len(corpus))
print("Queries per intent:")
print(pd.Series([label for _, label in corpus]).value_counts())

In [ ]:
taxonomy = {
"Understanding Diabetes": ["symptoms", "risk_factors"],
"Lifestyle Management": ["diet", "exercise"],
"Treatment and Monitoring": ["medication", "glucose_monitoring"],
"Complications and Emergencies": ["complications", "emergency"]
}
intent_category = {intent: category for category, intents in taxonomy.items() for intent in intents}

concepts = ["diabetes", "insulin", "glucose", "hba1c", "metformin", "hypoglycemia", "hyperglycemia", "neuropathy", "retinopathy", "kidney", "carbohydrate", "exercise"]
relationships = [
("insulin", "regulates", "glucose"), ("diabetes", "may cause", "hyperglycemia"),
("metformin", "helps manage", "glucose"), ("hba1c", "estimates average", "glucose"),
("exercise", "can improve sensitivity to", "insulin"), ("hypoglycemia", "means", "low glucose"),
("hyperglycemia", "means", "high glucose"), ("diabetes", "may damage", "kidney"),
("diabetes", "may cause", "neuropathy"), ("diabetes", "may cause", "retinopathy"),
("carbohydrate", "can raise", "glucose"), ("exercise", "can lower", "glucose")
]
print("Concepts:", len(concepts), "Relationships:", len(relationships))

In [ ]:
X = [q for q, label in corpus]
y = [label for q, label in corpus]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

vectorizer = FeatureUnion([
("word", TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2), sublinear_tf=True)),
("char", TfidfVectorizer(preprocessor=preprocess, analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True))
])
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
model = LogisticRegression(max_iter=2000, C=4, class_weight="balanced")
model.fit(X_train_vec, y_train)
predictions = model.predict(X_test_vec)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))
cm = confusion_matrix(y_test, predictions, labels=list(data.keys()))
display(pd.DataFrame(cm, index=data.keys(), columns=data.keys()))
ConfusionMatrixDisplay(cm, display_labels=list(data.keys())).plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
knowledge_base = {
"symptoms": "Common symptoms include increased thirst, frequent urination, increased hunger, fatigue, blurred vision, unexplained weight changes and slow-healing wounds. Some people have no noticeable symptoms. A blood test is needed for diagnosis.",
"risk_factors": "Risk factors include family history, excess weight, physical inactivity, increasing age, prediabetes, high blood pressure and a history of gestational diabetes. Having risk factors does not mean a person definitely has diabetes.",
"diet": "A balanced eating pattern can include non-starchy vegetables, pulses, whole grains, fruits in suitable portions and lean protein. Limit sugary drinks and highly refined foods. Individual meal plans should account for medicines, activity and personal health needs.",
"exercise": "Regular activity such as walking, cycling, swimming and strength training can support blood glucose management. Start gradually and follow medical advice, especially if you use insulin or medicines that can cause low blood sugar.",
"medication": "Diabetes treatment may include medicines such as metformin, other tablets or injectable medicines, including insulin. The choice depends on the type of diabetes and individual health. Do not start, stop or change a medicine or dose without your clinician.",
"glucose_monitoring": "A glucose meter measures blood glucose at a particular time. HbA1c estimates average blood glucose over roughly the previous three months. Your care team can explain the target range and how often you should test.",
"complications": "Over time, diabetes can affect the eyes, kidneys, nerves, heart and blood vessels. Blood glucose management, blood pressure and cholesterol control, foot care and regular eye and kidney checks can help reduce risks.",
"emergency": "A diabetes emergency can involve severe low blood sugar or a serious high blood sugar crisis. If a person is unconscious, having a seizure, severely confused, struggling to breathe or unable to swallow, call emergency services immediately. Do not give food or drink to someone who is unconscious or cannot swallow safely."
}

In [ ]:
emergency_phrases = ["unconscious", "unresponsive", "passed out", "cannot wake", "can't wake", "seizure", "unable to swallow", "cannot swallow", "can't swallow", "fruity breath", "ketoacidosis", "breathing rapidly", "difficulty breathing", "struggling to breathe"]
dosage_phrases = ["dose", "dosage", "how many units", "how much insulin", "insulin dose", "increase my insulin", "decrease my insulin", "change my insulin"]
unsupported_topics = ["cancer", "broken bone", "broken arm", "fracture", "asthma", "migraine", "toothache", "covid", "pregnancy test"]

def assistant(query, show_details=True):
    query = query.strip()
    q = query.lower()
    if not query:
        return "Please enter a question."
    if any(term in q for term in emergency_phrases):
        return "EMERGENCY ALERT\n" + knowledge_base["emergency"]
    if any(term in q for term in dosage_phrases):
        return "For safety, I cannot recommend a personal insulin or medicine dose. Follow your prescribed plan and contact your diabetes care team or pharmacist. If you may have taken too much insulin or feel severely unwell, seek urgent medical help."
    if any(term in q for term in unsupported_topics):
        return "I am designed for diabetes-related information and cannot reliably answer that topic. Please contact an appropriate healthcare professional."

    probabilities = model.predict_proba(vectorizer.transform([query]))[0]
    best = int(np.argmax(probabilities))
    intent = model.classes_[best]
    confidence = float(probabilities[best])
    processed = preprocess(query)
    domain_terms = ["diabetes", "diabetic", "glucose", "sugar", "insulin", "hba1c", "metformin", "blood", "hypoglycemia", "hyperglycemia", "sugar level", "blood sugar"]
    known_diabetes_context = any(term in q for term in domain_terms)

    if confidence < 0.30 or (not known_diabetes_context and confidence < 0.48):
        return "I am not confident that I understood the question. Please rephrase it as a diabetes-related question."

    answer = knowledge_base[intent]
    if show_details:
        return f"Intent: {intent}\nCategory: {intent_category[intent]}\nConfidence: {confidence:.1%}\n\n{answer}\n\nThis assistant provides general educational information, not a diagnosis or personal medical advice."
    return answer

In [ ]:
test_queries = [
"What signs show that someone may have diabetes?",
"Am I likely to get diabetes if my mother has it?",
"Which meals are best for a diabetic person?",
"Is cycling a good workout for people with diabetes?",
"How does metformin tablet work?",
"What is the average sugar test called?",
"Can diabetes damage the kidneys?",
"A diabetic person is unconscious",
"Can people with diabetes eat sweets?",
"How do I treat a broken arm?",
"Tell me about the history of computers",
"What dose of insulin should I take?"
]
for q in test_queries:
    print("You:", q)
    print("Assistant:", assistant(q))
    print("-" * 70)

In [ ]:
print("Diabetes Care Information Assistant")
print("Enter 'exit' to stop. You can ask multiple questions, one at a time.")
while True:
    query = input("\nYou: ").strip()
    if query.lower() in ["exit", "quit", "q"]:
        print("Goodbye")
        break
    print("\nAssistant:")
    print(assistant(query))